In [1]:
%pip install emoji pythainlp lightgbm

Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip install emoji pythainlp lightgbm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import emoji
import pandas as pd
import numpy as np
import joblib

from pythainlp.tokenize import word_tokenize
from pythainlp.corpus.common import thai_stopwords
from scipy.sparse import hstack

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC # ใช้ LinearSVC เพื่อความเร็วระดับแสง!
import lightgbm as lgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Loading data...")
df = pd.read_json("../../dataset/train_sentiment.json")
if "text" not in df.columns:
    df = df.transpose().reset_index(drop=True)
print(f"Data loaded successfully! Shape: {df.shape}")

Loading data...
Data loaded successfully! Shape: (9000, 2)


In [3]:
def clean_text_advanced(text):
    text = str(text)
    text = re.sub(r'&#\d+;', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' <URL> ', text)
    text = re.sub(r'@\S+', ' <USER> ', text)
    
    text = text.replace('❤', ' <POS_EMOJI> ').replace('👍', ' <POS_EMOJI> ')
    text = text.replace('👎', ' <NEG_EMOJI> ').replace('😡', ' <NEG_EMOJI> ')
    text = emoji.replace_emoji(text, replace=' <EMOJI> ')
    
    text = re.sub(r'5{3,}\+?', ' <LAUGH> ', text)
    text = re.sub(r'([ก-๙])\1{2,}', r'\1\1', text)
    text = re.sub(r'[^ก-๙a-zA-Z0-9\s_<>!]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Cleaning text...")
df["clean_text"] = df["text"].apply(clean_text_advanced)
print("Text cleaning complete!")

Cleaning text...
Text cleaning complete!


In [4]:
stopwords = set(thai_stopwords())
stopwords.discard('ไม่')
stopwords.discard('ดี')

def smart_tokenizer(text):
    tokens = word_tokenize(text, engine="newmm")
    result = []
    skip_next = False
    
    for i in range(len(tokens)):
        if skip_next:
            skip_next = False
            continue
        if tokens[i] == 'ไม่' and i + 1 < len(tokens) and tokens[i+1].strip() != '':
            result.append('ไม่_' + tokens[i+1])
            skip_next = True
        else:
            if tokens[i] not in stopwords and tokens[i].strip() != '':
                result.append(tokens[i])
    return result

X = df["clean_text"]
y = df["sentiment"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

print("Vectorizing text (Optimized Features)...")
# ลด Max Features เพื่อให้รันไวขึ้นโดยไม่เสียความแม่นยำมากนัก
tfidf_word = TfidfVectorizer(tokenizer=smart_tokenizer, ngram_range=(1, 2), min_df=3, max_features=10000)
X_word_train = tfidf_word.fit_transform(X_train)
X_word_test = tfidf_word.transform(X_test)

tfidf_char = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), min_df=3, max_features=10000)
X_char_train = tfidf_char.fit_transform(X_train)
X_char_test = tfidf_char.transform(X_test)

X_train_final = hstack([X_word_train, X_char_train])
X_test_final = hstack([X_word_test, X_char_test])
print(f"Vectorization complete! Features size: {X_train_final.shape[1]}")

Vectorizing text (Optimized Features)...


c:\Users\User\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Vectorization complete! Features size: 20000


In [5]:
print("Training Ensemble Model in Parallel...")

# ปรับ estimators ของ LightGBM ลงเหลือ 300 ก็พอครับ (รันไวกว่า 600 เท่าตัว)
model_lgb = lgb.LGBMClassifier(objective="multiclass", num_class=3, n_estimators=300, learning_rate=0.05, class_weight="balanced", random_state=42, n_jobs=-1)
model_lr = LogisticRegression(max_iter=500, C=1.0, class_weight="balanced", random_state=42, n_jobs=-1)
model_svm = LinearSVC(C=1.0, class_weight="balanced", random_state=42, max_iter=1000)

ensemble_model = VotingClassifier(
    estimators=[('lgb', model_lgb), ('lr', model_lr), ('svm', model_svm)],
    voting='hard',
    n_jobs=-1 # เทรน 3 โมเดลพร้อมกันรวดเดียว!
)

ensemble_model.fit(X_train_final, y_train)
print("Model training complete!")

Training Ensemble Model in Parallel...
Model training complete!


In [6]:
print("\nPredicting on Test Set...")
y_pred = ensemble_model.predict(X_test_final)

acc = accuracy_score(y_test, y_pred)
print("="*50)
print(f"🌟 FINAL MODEL ACCURACY: {acc:.4f} ({acc*100:.2f}%) 🌟")
print("="*50)

print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=[f"Actual_{c}" for c in le.classes_], columns=[f"Pred_{c}" for c in le.classes_])
print("\n🧩 Confusion Matrix:")
print(cm_df)

# บันทึกไฟล์พร้อมนำไปใช้ขึ้นเว็บ
joblib.dump(ensemble_model, "ensemble_model.pkl")
joblib.dump(tfidf_word, "tfidf_word.pkl")
joblib.dump(tfidf_char, "tfidf_char.pkl")
joblib.dump(le, "label_encoder.pkl")
print("\n✅ All models saved successfully! พร้อมเอาไปทำหน้าเว็บต่อแล้วครับ!")


Predicting on Test Set...
🌟 FINAL MODEL ACCURACY: 0.7800 (78.00%) 🌟

📊 Classification Report:
              precision    recall  f1-score   support

    negative       0.83      0.84      0.83       600
     neutral       0.71      0.68      0.69       600
    positive       0.79      0.83      0.81       600

    accuracy                           0.78      1800
   macro avg       0.78      0.78      0.78      1800
weighted avg       0.78      0.78      0.78      1800


🧩 Confusion Matrix:
                 Pred_negative  Pred_neutral  Pred_positive
Actual_negative            502            81             17
Actual_neutral              81           405            114
Actual_positive             20            83            497


c:\Users\User\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



✅ All models saved successfully! พร้อมเอาไปทำหน้าเว็บต่อแล้วครับ!


In [9]:
import re
import emoji
import joblib
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb

from pythainlp.tokenize import word_tokenize
from pythainlp.corpus.common import thai_stopwords
from scipy.sparse import hstack

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Suppress the "token_pattern" warning for a cleaner console
warnings.filterwarnings("ignore", category=UserWarning, module='sklearn')

# ---------------------------------------------------------
# 1. Load Dataset
# ---------------------------------------------------------
print("Loading data...")
# Adjust path if necessary
try:
    df = pd.read_json("../../dataset/train_sentiment.json")
    if "text" not in df.columns:
        df = df.transpose().reset_index(drop=True)
except Exception as e:
    print(f"Error loading JSON: {e}")
    exit()

# ---------------------------------------------------------
# 2. Advanced Preprocessing
# ---------------------------------------------------------
def clean_text_advanced(text):
    text = str(text)
    text = re.sub(r'&#\d+;', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' <URL> ', text)
    text = re.sub(r'@\S+', ' <USER> ', text)
    
    # Emoji Mapping
    text = text.replace('❤', ' <POS_EMOJI> ').replace('👍', ' <POS_EMOJI> ')
    text = text.replace('👎', ' <NEG_EMOJI> ').replace('😡', ' <NEG_EMOJI> ')
    text = emoji.replace_emoji(text, replace=' <EMOJI> ')
    
    # Laugh Processing
    text = re.sub(r'5{3,}\+?', ' <LAUGH> ', text)
    
    # Vowel Stretching Normalization
    text = re.sub(r'([ก-๙])\1{2,}', r'\1\1', text)
    
    # Keep specific markers
    text = re.sub(r'[^ก-๙a-zA-Z0-9\s_<>!]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Cleaning text...")
df["clean_text"] = df["text"].apply(clean_text_advanced)

# ---------------------------------------------------------
# 3. Smart Tokenizer & Negation Binding
# ---------------------------------------------------------
stopwords = set(thai_stopwords())
stopwords.discard('ไม่')
stopwords.discard('ดี') 

def smart_tokenizer(text):
    tokens = word_tokenize(text, engine="newmm")
    result = []
    skip_next = False
    
    for i in range(len(tokens)):
        if skip_next:
            skip_next = False
            continue
            
        # Negation Handling
        if tokens[i] == 'ไม่' and i + 1 < len(tokens) and tokens[i+1].strip() != '':
            bound_word = 'ไม่_' + tokens[i+1]
            result.append(bound_word)
            skip_next = True
        else:
            if tokens[i] not in stopwords and tokens[i].strip() != '':
                result.append(tokens[i])
    return result

# ---------------------------------------------------------
# 4. Feature Extraction & Data Splitting
# ---------------------------------------------------------
X = df["clean_text"]
y = df["sentiment"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print("Vectorizing text (TF-IDF)...")
# Word Level - Added lowercase=False to preserve your <TAGS>
tfidf_word = TfidfVectorizer(tokenizer=smart_tokenizer, ngram_range=(1, 2), min_df=3, max_features=40000, lowercase=False)
X_word_train = tfidf_word.fit_transform(X_train)
X_word_test = tfidf_word.transform(X_test)

# Character Level
tfidf_char = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), min_df=3, max_features=20000)
X_char_train = tfidf_char.fit_transform(X_train)
X_char_test = tfidf_char.transform(X_test)

X_train_final = hstack([X_word_train, X_char_train])
X_test_final = hstack([X_word_test, X_char_test])

# ---------------------------------------------------------
# 5. Optimized Ensemble Model
# ---------------------------------------------------------
print("Training Ensemble Model (LGBM + LogReg + Fast SVM)...")

# LightGBM
model_lgb = lgb.LGBMClassifier(
    objective="multiclass", num_class=3, n_estimators=500, 
    learning_rate=0.05, class_weight="balanced", random_state=42, 
    n_jobs=-1, verbose=-1
)

# Logistic Regression
model_lr = LogisticRegression(
    max_iter=1000, C=1.0, class_weight="balanced", 
    random_state=42, n_jobs=-1
)

# Optimized SVM using LinearSVC + Calibration for probabilities
base_svm = LinearSVC(C=1.0, class_weight="balanced", random_state=42, max_iter=2000)
model_svm_fast = CalibratedClassifierCV(base_svm, cv=3)

# Ensemble with n_jobs=-1 to train models in parallel
ensemble_model = VotingClassifier(
    estimators=[('lgb', model_lgb), ('lr', model_lr), ('svm', model_svm_fast)],
    voting='soft',
    n_jobs=-1
)

ensemble_model.fit(X_train_final, y_train)

# ---------------------------------------------------------
# 6. Evaluation & Results
# ---------------------------------------------------------
print("\nPredicting on Test Set...")
y_pred = ensemble_model.predict(X_test_final)

acc = accuracy_score(y_test, y_pred)
print("="*50)
print(f"🌟 FINAL MODEL ACCURACY: {acc:.4f} ({acc*100:.2f}%) 🌟")
print("="*50)

print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Save Models
print("\nSaving models...")
joblib.dump(ensemble_model, "ensemble_model.pkl")
joblib.dump(tfidf_word, "tfidf_word.pkl")
joblib.dump(tfidf_char, "tfidf_char.pkl")
joblib.dump(le, "label_encoder.pkl")

print("✅ All processes completed successfully!")

Loading data...
Cleaning text...
Vectorizing text (TF-IDF)...
Training Ensemble Model (LGBM + LogReg + Fast SVM)...

Predicting on Test Set...
🌟 FINAL MODEL ACCURACY: 0.7933 (79.33%) 🌟

📊 Classification Report:
              precision    recall  f1-score   support

    negative       0.84      0.84      0.84       600
     neutral       0.73      0.70      0.71       600
    positive       0.81      0.84      0.82       600

    accuracy                           0.79      1800
   macro avg       0.79      0.79      0.79      1800
weighted avg       0.79      0.79      0.79      1800


Saving models...
✅ All processes completed successfully!
